# 04 Loss of the organisation with acquisition quality
The same analysis after degrading each subject's data: fewer directions, larger voxels, clinical-grade protocols. Classical methods only (networks: notebook 06).

In [ ]:
DEV_MODE = True    # True: small test run (dev: block of config.yaml). False: full experiment (full: block).

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from retinopath.core import load_config, get_device, seed_all, Timer
from retinopath import pipeline as PL

cfg = load_config(DEV_MODE); device = get_device(); seed_all(cfg.seed)
RES = cfg.P.out / "results"; FIG = cfg.P.out / "figures"; RES.mkdir(exist_ok=True); FIG.mkdir(exist_ok=True)
print(f"DEV_MODE={DEV_MODE} | device={device} | outputs -> {cfg.P.out}")
if DEV_MODE: print("DEV_MODE: small test run. Its outputs are kept apart in outputs/dev and are not experiment results.")

In [ ]:
subjects = PL.make_subject_lists(cfg); test = [PL.session(s) for s in subjects["test"]]
protos = [p for p in cfg.run_protocols if p != "REF"]; clin = [p for p in cfg.clinical_protocols if p in protos]
combos = [(p, "csd") for p in protos]                                      # CSD everywhere
combos += [(p, m) for p in clin for m in ("rumba", "csd_mppca", "csd_interp")]   # slower baselines: clinical protocols only
combos += [("REF", "msmt_csd+prob")] + [(p, "csd+prob") for p in clin]      # same fODFs, probabilistic tracking instead of PFT
combos += [("REF", "rumba")]
with Timer(cfg, "04_protocol_map"):
    df = PL.run_grid(cfg, test, combos, "protocols")
df.groupby(["protocol", "method"])[["n_pairs", "rsi_ree_all", "atlas_rsi_ree_all", "acc", "peak_err_deg", "wdice_ref", "badj_ref_mm"]].median().round(3)

In [ ]:
ref = pd.read_csv(RES / "h1_reference.csv").query("method == 'msmt_csd'")
t = pd.concat([ref, df]).pivot_table(index="method", columns="protocol", values="rsi_ree_all", aggfunc="median")
fig, ax = plt.subplots(figsize=(1.2 * t.shape[1] + 2, 0.6 * t.shape[0] + 1.5))
im = ax.imshow(t.to_numpy(), cmap="viridis"); ax.set_xticks(range(t.shape[1]), t.columns); ax.set_yticks(range(t.shape[0]), t.index)
for (i, j), v in np.ndenumerate(t.to_numpy()):
    if np.isfinite(v): ax.text(j, i, f"{v:.2f}", ha="center", va="center", color="w")
fig.colorbar(im, label="median RSI"); fig.savefig(FIG / "protocol_map.png", dpi=150, bbox_inches="tight")